In [1]:
import numpy as np
import axelrod
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from axelrod.action import Action, actions_to_str
from axelrod.player import Player
from axelrod.strategy_transformers import (
    FinalTransformer,
    TrackHistoryTransformer,
)
import random
from copy import deepcopy
from collections import Counter


In [ ]:
PARAM_KEYS = [
    # cooperation
    'coop_low_a',   'coop_low_b',   'coop_low_c',
    'coop_med_a',   'coop_med_b',   'coop_med_c',
    'coop_high_a',  'coop_high_b',  'coop_high_c',
    # adaptivity
    'adap_no_a',    'adap_no_b',    'adap_no_c',
    'adap_yes_a',   'adap_yes_b',   'adap_yes_c',
    # forgiveness
    'forg_sigma',
    'forg_med_a',   'forg_med_b',   'forg_med_c',
    'forg_high_a',  'forg_high_b',  'forg_high_c',
    # stochastic
    'stoch_none_a', 'stoch_none_b', 'stoch_none_c',
    'stoch_some_a', 'stoch_some_b', 'stoch_some_c',
    'stoch_alw_a',  'stoch_alw_b',  'stoch_alw_c',
    # output
    'D_a', 'D_b', 'D_c',
    'C_a', 'C_b', 'C_c',
    # thresholds
    'd_threshold', 'c_threshold',
]

# Bounds for each parameter — (min, max)
BOUNDS = [
    # cooperation low
    (0, 40), (0, 50), (20, 60),
    # cooperation medium
    (15, 45), (30, 70), (50, 85),
    # cooperation high
    (50, 80), (65, 99), (85, 99),
    # adaptivity no
    (0, 30), (0, 50), (25, 65),
    # adaptivity yes
    (35, 70), (55, 99), (75, 99),
    # forgiveness
    (5, 40),                         # sigma
    (5, 40), (30, 70), (60, 95),     # medium
    (55, 85), (70, 99), (85, 99),    # high
    # stochastic none
    (0, 25), (0, 35), (15, 55),
    # stochastic sometimes
    (15, 50), (35, 70), (50, 85),
    # stochastic always
    (50, 80), (65, 99), (85, 99),
    # output D
    (0, 25), (10, 45), (30, 65),
    # output C
    (20, 55), (50, 85), (75, 99),
    # thresholds
    (0.1, 0.8), (0.2, 0.9),
]

YOUR_BASELINE = [
    # cooperation — automf evenly spaced
    0, 0, 49,   0, 49, 99,   49, 99, 99,
    # adaptivity
    0, 0, 99,   0, 99, 99,
    # forgiveness
    25,                      # sigma
    25, 50, 75,              # medium
    49, 99, 99,              # high
    # stochastic
    0, 0, 49,   0, 49, 99,   49, 99, 99,
    # output
    0, 25, 50,   35, 75, 99,
    # thresholds
    0.4, 0.6,
]

In [ ]:
def decode(individual):
    """Convert flat list back to named params dict."""
    return dict(zip(PARAM_KEYS, individual))


def repair(individual):
    """
    Clip to bounds and enforce a <= b <= c ordering
    within each trimf triplet.
    """
    ind = individual.copy()

    # Clip everything to its bounds first
    for i, (lo, hi) in enumerate(BOUNDS):
        ind[i] = float(np.clip(ind[i], lo, hi))

    # Triplet indices (a, b, c) that must be ordered
    triplets = [
        (0,1,2), (3,4,5), (6,7,8),         # cooperation
        (9,10,11), (12,13,14),              # adaptivity
        (16,17,18), (19,20,21),             # forgiveness medium, high
        (22,23,24), (25,26,27), (28,29,30), # stochastic
        (31,32,33), (34,35,36),             # output
    ]

    for a, b, c in triplets:
        ind[b] = max(ind[a], ind[b])
        ind[c] = max(ind[b], ind[c])

    return ind

In [4]:
C, D = Action.C, Action.D

class FuzzyMethods():
    @staticmethod
    def calcCooperation(self, opponent):
        return (Counter(opponent.history)[C])/len(opponent.history)*100
    
    @staticmethod
    def calcAdaptivity(self, opponent):
        adapCounter = 0
        adapReaction = 0

        if(len(self.history) < 3): 
            return 0
        
        for i in range(3, len(self.history)):
            if (self.history[i-3] == C and self.history[i-2] == D):
                adapCounter += 1
                if (opponent.history[i-1] == D):
                    adapReaction += 1
                elif (self.history[i-1] == D and opponent.history[i] == D):
                    adapReaction += 0.5
            elif (self.history[i-3] == D and self.history[i-2] == C):
                adapCounter += 1
                if (opponent.history[i-1] == C):
                    adapReaction += 1
                elif (self.history[i-1] == C and opponent.history[i] == C):
                    adapReaction += 0.5

        if adapCounter == 0:
            return 0
        
        return adapReaction/adapCounter*100

    
    @staticmethod
    def calcForgiveness(self, opponent):
        DCounter = 0
        punishmentCounter = 0

        for i in range(0, len(self.history)-1):
            if(self.history[i] == D):
                DCounter += 1
                for j in range (i+1, len(opponent.history)):
                    if(opponent.history[j] == C):
                        break
                    else:
                        punishmentCounter += 1
              

        if punishmentCounter > 0:
            return DCounter/punishmentCounter*100
        else:
            return 100
    
    @staticmethod
    def calcStochastic(self, opponent):
        patterns = [
            [C, C, C],
            [C, C, D],
            [C, D, C],
            [C, D, D],
            [D, C, C],
            [D, C, D],
            [D, D, C],
            [D, D, D]
        ]

        non_stochasticCounter = 0
        patternPlayedCounter = 0

        for p in patterns:
            opponentsReactions = []
            for i in range(0, len(self.history)-3):
                if ([self.history[i], self.history[i+1], self.history[i+2]] == p):
                    opponentsReactions.append([opponent.history[i+1], opponent.history[i+2], opponent.history[i+3]])
            
            unique_patterns = len(set(tuple(sub) for sub in opponentsReactions))

            if(len(opponentsReactions) > 0):
                non_stochasticCounter += 0 if unique_patterns == 1 else unique_patterns
                patternPlayedCounter += len(opponentsReactions)
        
        if patternPlayedCounter == 0:
            return 0
        
        return non_stochasticCounter/patternPlayedCounter*100
    

In [ ]:
def build_player(params):
    """Build a fresh OptimizedFuzzy player from params dict."""

    _cooperation = ctrl.Antecedent(np.arange(0, 100, 1), 'cooperation')
    _adaptivity  = ctrl.Antecedent(np.arange(0, 100, 1), 'adaptivity')
    _forgiveness = ctrl.Antecedent(np.arange(0, 100, 1), 'forgiveness')
    _stochastic  = ctrl.Antecedent(np.arange(0, 100, 1), 'stochastic')

    _cooperation['low']    = fuzz.trimf(_cooperation.universe, [params['coop_low_a'],  params['coop_low_b'],  params['coop_low_c']])
    _cooperation['medium'] = fuzz.trimf(_cooperation.universe, [params['coop_med_a'],  params['coop_med_b'],  params['coop_med_c']])
    _cooperation['high']   = fuzz.trimf(_cooperation.universe, [params['coop_high_a'], params['coop_high_b'], params['coop_high_c']])

    _adaptivity['no']  = fuzz.trimf(_adaptivity.universe, [params['adap_no_a'],  params['adap_no_b'],  params['adap_no_c']])
    _adaptivity['yes'] = fuzz.trimf(_adaptivity.universe, [params['adap_yes_a'], params['adap_yes_b'], params['adap_yes_c']])

    _forgiveness['low']    = fuzz.gaussmf(_forgiveness.universe, 0, params['forg_sigma'])
    _forgiveness['medium'] = fuzz.trimf(_forgiveness.universe,   [params['forg_med_a'],  params['forg_med_b'],  params['forg_med_c']])
    _forgiveness['high']   = fuzz.trimf(_forgiveness.universe,   [params['forg_high_a'], params['forg_high_b'], params['forg_high_c']])

    _stochastic['none']      = fuzz.trimf(_stochastic.universe, [params['stoch_none_a'], params['stoch_none_b'], params['stoch_none_c']])
    _stochastic['sometimes'] = fuzz.trimf(_stochastic.universe, [params['stoch_some_a'], params['stoch_some_b'], params['stoch_some_c']])
    _stochastic['always']    = fuzz.trimf(_stochastic.universe, [params['stoch_alw_a'],  params['stoch_alw_b'],  params['stoch_alw_c']])

    _resulting_strategy = ctrl.Consequent(np.arange(0, 100, 1), 'resulting_strategy')
    _resulting_strategy['D'] = fuzz.trimf(_resulting_strategy.universe, [params['D_a'], params['D_b'], params['D_c']])
    _resulting_strategy['C'] = fuzz.trimf(_resulting_strategy.universe, [params['C_a'], params['C_b'], params['C_c']])

    rule_default = ctrl.Rule(_cooperation['low'] | _cooperation['medium'] | _cooperation['high'], _resulting_strategy['C'])
    rule1 = ctrl.Rule(_cooperation['high'] & _adaptivity['no'] & (_forgiveness['medium'] | _forgiveness['high']), _resulting_strategy['D'])
    rule2 = ctrl.Rule(_forgiveness['low'] & _cooperation['high'], _resulting_strategy['C'])
    rule3 = ctrl.Rule(_stochastic['always'] | _adaptivity['no'], _resulting_strategy['D'])
    rule4 = ctrl.Rule(_cooperation['low'] | (_cooperation['medium'] & _forgiveness['low']), _resulting_strategy['D'])
    rule5 = ctrl.Rule(_cooperation['medium'] & _forgiveness['medium'] & _adaptivity['yes'], _resulting_strategy['C'])

    _chosen_strategy = ctrl.ControlSystemSimulation(ctrl.ControlSystem([rule_default, rule1, rule2, rule3, rule4, rule5]))

    class PSOFuzzy(Player):
        name = "PSOFuzzy"
        classifier = {"memory_depth": float("inf"), "stochastic": False,
                      "long_run_time": False, "inspects_source": False,
                      "manipulates_source": False, "manipulates_state": False}

        cooperation        = _cooperation
        adaptivity         = _adaptivity
        forgiveness        = _forgiveness
        stochastic         = _stochastic
        resulting_strategy = _resulting_strategy
        chosen_strategy    = _chosen_strategy
        d_thresh           = params['d_threshold']
        c_thresh           = params['c_threshold']
        first_time         = True
        h                  = {'Name': '', 'Fuzzy': [], 'Opponent': []}

        def strategy(self, opponent: Player) -> Action:
            if len(self.history) == 0 or D not in opponent.history:
                return C
            coop  = FuzzyMethods.calcCooperation(self, opponent)
            adap  = FuzzyMethods.calcAdaptivity(self, opponent)
            forg  = FuzzyMethods.calcForgiveness(self, opponent)
            stoch = FuzzyMethods.calcStochastic(self, opponent)
            self.chosen_strategy.input['cooperation'] = coop
            self.chosen_strategy.input['adaptivity']  = adap
            self.chosen_strategy.input['forgiveness'] = forg
            self.chosen_strategy.input['stochastic']  = stoch
            try:
                self.chosen_strategy.compute()
                output_val   = self.chosen_strategy.output['resulting_strategy']
                d_membership = fuzz.interp_membership(self.resulting_strategy.universe, self.resulting_strategy['D'].mf, output_val)
                c_membership = fuzz.interp_membership(self.resulting_strategy.universe, self.resulting_strategy['C'].mf, output_val)
                if d_membership >= self.d_thresh and c_membership < self.c_thresh:
                    return D
            except:
                return C
            return C

    return PSOFuzzy()



In [6]:
def evaluate(individual):
    params = decode(repair(individual))
    try:
        player    = build_player(params)
        opponents = [player() for player in axelrod.stewart_plotkin_strategies]
        results   = axelrod.Tournament([player] + opponents, turns=200, repetitions=3).play(progress_bar=False)
        return np.mean(results.normalised_scores[0])
    except Exception as e:
        print(f"  [evaluate ERROR] {type(e).__name__}: {e}")
        return 0.0

In [ ]:
# Reuse PARAM_KEYS, BOUNDS, YOUR_BASELINE, decode, repair, build_player, evaluate
# from the GA file — put them in a shared shared_utils.py and import from there


# ─────────────────────────────────────────────
#  PARTICLE SWARM OPTIMIZATION
# ─────────────────────────────────────────────
#
#  Each particle has:
#    position  — current parameter values (the solution)
#    velocity  — how fast/which direction it's moving
#    pbest     — best position this particle personally found
#    gbest     — best position ANY particle found (shared)
#
#  Update rules each iteration:
#    velocity = w * velocity
#             + c1 * r1 * (pbest - position)   ← pull toward personal best
#             + c2 * r2 * (gbest - position)   ← pull toward global best
#    position = position + velocity


class Particle:

    def __init__(self, position):
        self.position  = np.array(position, dtype=float)
        self.velocity  = np.array([
            np.random.uniform(-(hi - lo) * 0.1, (hi - lo) * 0.1)
            for lo, hi in BOUNDS
        ])
        self.pbest          = self.position.copy()
        self.pbest_score    = -np.inf

    def update_velocity(self, gbest, w, c1, c2):
        r1 = np.random.uniform(0, 1, size=len(self.position))
        r2 = np.random.uniform(0, 1, size=len(self.position))

        cognitive = c1 * r1 * (self.pbest   - self.position)
        social    = c2 * r2 * (gbest        - self.position)

        self.velocity = w * self.velocity + cognitive + social

        # Clamp velocity to 20% of range to prevent explosion
        for i, (lo, hi) in enumerate(BOUNDS):
            max_v            = (hi - lo) * 0.2
            self.velocity[i] = np.clip(self.velocity[i], -max_v, max_v)

    def update_position(self):
        self.position = repair(list(self.position + self.velocity))
        self.position = np.array(self.position)


def run_pso(
    n_particles  = 30,
    n_iterations = 100,
    w            = 0.7,   # inertia weight — how much old velocity is kept
    c1           = 1.5,   # cognitive coefficient — trust in personal best
    c2           = 1.5,   # social coefficient — trust in global best
    w_decay      = 0.99,  # slowly reduce inertia to shift from explore to exploit
):
    print("Initialising swarm...")

    # Initialise particles — seed one with your baseline
    particles    = [Particle(YOUR_BASELINE.copy())]
    particles   += [Particle([np.random.uniform(lo, hi) for lo, hi in BOUNDS])
                    for _ in range(n_particles - 1)]

    gbest        = particles[0].position.copy()
    gbest_score  = -np.inf

    for iteration in range(n_iterations):

        for particle in particles:

            score = evaluate(list(particle.position))

            # Update personal best
            if score > particle.pbest_score:
                particle.pbest_score = score
                particle.pbest       = particle.position.copy()

            # Update global best
            if score > gbest_score:
                gbest_score = score
                gbest       = particle.position.copy()

        # Update velocities and positions
        for particle in particles:
            particle.update_velocity(gbest, w, c1, c2)
            particle.update_position()

        # Decay inertia — more exploitation as iterations progress
        w *= w_decay

        scores = [p.pbest_score for p in particles]
        print(f"Iter {iteration+1:>4}/{n_iterations} | GBest: {gbest_score:.4f} | Swarm avg pbest: {np.mean(scores):.4f} | w: {w:.4f}")

    print(f"\n=== PSO COMPLETE ===")
    print(f"Best score: {gbest_score:.4f}")
    print(f"Best params: {decode(list(gbest))}")
    return gbest, gbest_score


if __name__ == "__main__":
    best_pos, best_score = run_pso()

Initialising swarm...
